# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a practical walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library, following a structured approach from data loading to visualization.

### Dataset Source
The dataset is defined by a Croissant schema and is accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"\nPublished: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Examine available record sets, their `@id`s, field `@id`s and structure.

In [ ]:
# List all record sets by `@id` and field `@id`s
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets defined in the schema.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs.id}")
        if hasattr(rs, 'fields'):
            for field in rs.fields:
                print(f"  - Field: {field.id} (name: {field.name}, type: {getattr(field, 'data_type', 'unknown')})")
        print("")

Let's display the first record from each record set using their `@id`.

In [ ]:
# Print the first record from each record set
for rs in dataset.record_sets:
    print(f"\nRecord Set @id: {rs.id}")
    try:
        recs = list(dataset.records(record_set=rs.id))
        if recs:
            print("Example Record:")
            pprint.pprint(recs[0])
        else:
            print("No records found.")
    except Exception as e:
        print(f"Error reading records: {e}")

## 3. Data Extraction

We'll extract data from all main record sets using their `@id`.

**Note:** All further references to record sets and fields will use their `@id` fields as per FAIR principles.

In [ ]:
# Collect all record set IDs
record_set_ids = [rs.id for rs in dataset.record_sets]
if not record_set_ids:
    print("No record sets found in Croissant schema.")
else:
    print("Record set @ids:")
    for rs_id in record_set_ids:
        print(f"- {rs_id}")

In [ ]:
# Extract all dataframes by record set @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    nrows = dataframes[record_set_id].shape[0]
    print(f"Loaded {nrows} records for {record_set_id}")
    if nrows > 0:
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")

In [ ]:
# Preview the first few records of each dataframe
for record_set_id, df in dataframes.items():
    print(f"\nFirst 5 records from record set: {record_set_id}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping or summarizing key attributes.

**We'll perform EDA on one main record set.**

In [ ]:
# Select a record set for analysis (choose first if unsure)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id is None:
    raise ValueError("No record sets to analyze.")
df = dataframes[main_record_set_id]
print(f"Analyzing record set: {main_record_set_id}")

# Identify candidate numeric fields (by trying to convert columns)
numeric_candidates = []
for col in df.columns:
    try:
        pd.to_numeric(df[col].dropna().iloc[0])
        numeric_candidates.append(col)
    except Exception:
        continue
print(f"Possible numeric fields: {numeric_candidates}")

# Select first numeric field for demonstration (update index as needed)
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # Use column name, which is field @id
else:
    print("No numeric fields found to process.")
    numeric_field_id = None

# Filtering and normalization example (if numeric field exists)
if numeric_field_id:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Choose a threshold for demonstration (median)
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (using field @id):")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Identify candidate group fields (object or category type, ex: string columns)
    group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
    print(f"Suggested group fields: {group_candidates}")
    
    # Group by chosen field (use first found, if available)
    if group_candidates:
        group_field_id = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id} (using @id):")
        display(grouped_df.head())
else:
    print("Skipping numeric analysis, no numeric field found.")

## 5. Visualization
Visualize the distribution of the selected numeric field and optionally its grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'Mean of {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=65)
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion

In this notebook, we demonstrated how to explore and process a FAIR² clinical dataset defined by a Croissant schema, using the `mlcroissant` library. All data manipulations and references used the canonical `@id` of record sets and fields, ensuring full traceability with the Croissant metadata standard.

We loaded metadata, browsed available fields, filtered and normalized numeric values using their field `@id`, and visualized key distributions. This approach may be adapted for further analyses, statistical modeling, or integration in data science workflows leveraging rich, structured metadata provided by Croissant schemas.